# 1단계: 차량 탐지 모델 파인튜닝 — v3 (문제 수정판)

**v2 → v3 변경 사항 (실제 라벨 json `보강센트럴빌_2차_앞_BC2000203.json` 구조 확인 후 반영):**

| # | 문제 | 수정 |
|---|---|---|
| A | Windows에서 `cv2.imread`가 한글 경로를 못 읽어 조용히 대량 스킵 + 크기 확인용 전체 디코딩 낭비 | 이미지 크기는 **meta의 resolution 코드(FHD 등) → 크기 매핑**으로 해결. 매핑에 없으면 카메라당 1장만 한글경로 대응 함수(`imread_kr`)로 읽어 캐시 |
| B | 랜덤 8:2 분리 → 같은 카메라의 연속 프레임이 train/val에 섞이는 **데이터 누수** | **카메라 단위 분리** (val 카메라의 프레임은 train에 절대 안 들어감) |
| C | `epochs=5` (patience=10이 무의미) | `epochs=80, patience=15` + 스모크 테스트 셀 별도 제공 |
| D | 경계를 살짝 벗어난 bbox를 통째로 폐기 (실측: 최대 좌표 1922×1081로 FHD 경계 초과 박스 실존) | **클리핑 후 사용**, 클리핑 뒤 2px 미만으로 소멸한 박스만 제외 |
| E | 카메라 간 동일 파일명 충돌 시 이미지–라벨 불일치 가능 | 출력 파일명에 **카메라ID 접두어** 부여 |
| — | 기존 결과물과 충돌 | 출력을 전부 새 경로로: `yolo_dataset_v3/`, run 이름 `vehicle_detector_v3`, 최종 모델 `models/stage1_vehicle_detector_v3.pt` |

- 데이터: AI-Hub "교통문제 해결을 위한 CCTV 교통 영상(시내도로)" (dataSetSn=165)
- 클래스: 승용차, 소형버스, 대형버스, 트럭, 대형 트레일러, 오토바이(자전거), 보행자 (7종, `분류없음` 제외)
- bbox: `[xmin, ymin, xmax, ymax]` 픽셀 좌표, annotation 1개 = 이미지 1장 (`category_id` 리스트와 `bbox` 리스트를 zip)


## 0. 환경 확인

In [1]:
import os, sys, json, shutil, random
from pathlib import Path
from collections import defaultdict, Counter

import cv2
import numpy as np
import torch
from tqdm import tqdm

print("Python:", sys.version)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


In [2]:
# 처음 한 번만 실행 (이미 설치되어 있으면 생략)
# !pip install ultralytics

from ultralytics import YOLO


## 0-1. 한글 경로 대응 이미지 로더 (수정 A)

Windows의 `cv2.imread`는 한글이 포함된 경로에서 `None`을 반환합니다.
이 노트북에서 이미지를 읽는 모든 곳은 아래 함수를 사용합니다.

In [3]:
def imread_kr(path):
    """한글/유니코드 경로에서도 동작하는 이미지 로드. 실패 시 None."""
    try:
        data = np.fromfile(str(path), dtype=np.uint8)
        if data.size == 0:
            return None
        return cv2.imdecode(data, cv2.IMREAD_COLOR)
    except Exception:
        return None


## 1. 데이터 경로 설정

스크린샷 기준 실제 구조:
- 이미지: `...\교통문제 해결을 위한 CCTV 교통 영상(시내도로)\Training\교통안전(Bbox)\[원천]{장소}\{장소}\{카메라ID}\*.jpg`
- 라벨: `...\101.교통문제 해결을 위한 CCTV 교통 데이터(시내도로)\01.데이터\1.Training\교통안전(Bbox)\1.라벨링데이터_230510_add\1.라벨링데이터\{장소}\{카메라ID}\{장소}_{카메라ID}.json`

Tracking/Segmentation까지 쓰려면 리스트에 경로를 추가하면 됩니다.
(지금 단계는 Bbox만 권장 — 라벨 구조가 카테고리마다 다를 수 있음)

In [4]:
# ==== 실제 경로로 수정하세요 ====
IMAGE_ROOTS = [
    Path(r"C:\Users\Win11Pro\Downloads\교통문제 해결을 위한 CCTV 교통 영상(시내도로)\Training\교통안전(Bbox)"),
]
LABEL_ROOTS = [
    Path(r"C:\Users\Win11Pro\Downloads\101.교통문제 해결을 위한 CCTV 교통 데이터(시내도로)\01.데이터\1.Training\교통안전(Bbox)\1.라벨링데이터_230510_add\1.라벨링데이터"),
]
# ==== 새 출력 경로 (기존 yolo_dataset과 절대 겹치지 않음) ====
OUTPUT_ROOT = Path("./yolo_dataset_v3")
RUN_PROJECT = "flood_stage1"
RUN_NAME    = "vehicle_detector_v3"
FINAL_MODEL_PATH = Path("./models/stage1_vehicle_detector_v3.pt")
# ============================================================

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

def find_all_files(roots, exts: set):
    files = []
    for root in roots:
        if not root.exists():
            print(f"[경고] 경로가 존재하지 않습니다: {root}")
            continue
        files.extend(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in exts)
    return files

image_files = find_all_files(IMAGE_ROOTS, IMAGE_EXTS)
label_json_files = find_all_files(LABEL_ROOTS, {".json"})

print(f"이미지 파일: {len(image_files):,}개")
print(f"라벨 json 파일: {len(label_json_files):,}개 (카메라 1대당 1개)")

# (수정 E 사전 점검) 카메라 간 파일명 충돌 실태 확인
dup = len(image_files) - len({p.name for p in image_files})
print(f"카메라 간 동일 파일명: {dup:,}건 → v3는 카메라ID 접두어로 자동 회피")


이미지 파일: 145,694개
라벨 json 파일: 70개 (카메라 1대당 1개)
카메라 간 동일 파일명: 33,685건 → v3는 카메라ID 접두어로 자동 회피


## 2. 라벨 인덱스 + 이미지 크기 인덱스 구축 (수정 A)

- `label_index`: camera_id → {파일명: [(category_id, bbox), ...]}
- `size_index`: camera_id → {파일명: (width, height)}
  - json의 `images`에는 width/height가 **없음** (실측 확인). 대신 각 image가 `meta_id`로 meta와 연결되고, meta에 `resolution` 코드(FHD 등)가 있음
  - resolution 코드 → 크기 매핑으로 해결, 미지의 코드면 그 카메라 이미지 1장만 실제로 읽어 확인

In [5]:
RES_MAP = {"FHD": (1920, 1080), "HD": (1280, 720), "UHD": (3840, 2160), "4K": (3840, 2160)}
EXCLUDE_CATEGORY_NAMES = {"분류없음"}

label_index = {}    # camera_id -> {basename: [(cat_id, bbox), ...]}
size_index  = {}    # camera_id -> {basename: (w, h)}
unknown_res = Counter()

global_categories = None
mismatch_warned = False

for jp in tqdm(label_json_files, desc="라벨 json 로딩중"):
    try:
        with open(jp, encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        print(f"[경고] json 읽기 실패, 건너뜀: {jp} ({e})")
        continue

    cats = {c["id"]: c["name"] for c in data.get("categories", [])}
    if global_categories is None:
        global_categories = cats
    elif cats != global_categories and not mismatch_warned:
        print(f"[경고] 카테고리 구성이 다른 json 발견: {jp}")
        mismatch_warned = True

    meta_list = data.get("meta", [])
    camera_id = meta_list[0]["camera_id"] if meta_list else jp.stem.split("_")[-1]
    meta_res = {m["id"]: m.get("resolution", "") for m in meta_list}

    cam_lbl = label_index.setdefault(camera_id, {})
    cam_sz  = size_index.setdefault(camera_id, {})

    image_id_to_name = {}
    for img in data.get("images", []):
        basename = img["file_name"].split("/")[-1]
        image_id_to_name[img["id"]] = basename
        res_code = meta_res.get(img.get("meta_id"), "")
        wh = RES_MAP.get(res_code)
        if wh is None:
            unknown_res[res_code or "(없음)"] += 1
        else:
            cam_sz[basename] = wh

    for ann in data.get("annotations", []):
        basename = image_id_to_name.get(ann["image_id"])
        if basename is None:
            continue
        boxes = []
        for cat_id, bbox in zip(ann["category_id"], ann["bbox"]):
            if global_categories.get(cat_id, "") in EXCLUDE_CATEGORY_NAMES:
                continue
            boxes.append((cat_id, bbox))
        if boxes:
            cam_lbl[basename] = boxes

print(f"인덱싱된 카메라 수: {len(label_index)}")
print("카테고리:", global_categories)
if unknown_res:
    print("[안내] RES_MAP에 없는 resolution 코드 발견(해당 이미지는 변환 시 1장 실측으로 처리):", dict(unknown_res))


라벨 json 로딩중: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 70/70 [00:13<00:00,  5.32it/s]

인덱싱된 카메라 수: 70
카테고리: {1: '승용차', 2: '소형버스', 3: '대형버스', 4: '트럭', 5: '대형 트레일러', 6: '오토바이(자전거)', 7: '보행자', 8: '분류없음'}
[안내] RES_MAP에 없는 resolution 코드 발견(해당 이미지는 변환 시 1장 실측으로 처리): {'(없음)': 11}


## 3. 클래스 매핑 (`분류없음` 제외, category_id 오름차순 고정)

In [6]:
used_cat_ids = sorted(cid for cid in global_categories
                      if global_categories[cid] not in EXCLUDE_CATEGORY_NAMES)
CAT_ID_TO_YOLO_ID = {cid: i for i, cid in enumerate(used_cat_ids)}
CLASS_NAMES = [global_categories[cid] for cid in used_cat_ids]

print("YOLO 클래스 순서:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {i}: {name}")


YOLO 클래스 순서:
  0: 승용차
  1: 소형버스
  2: 대형버스
  3: 트럭
  4: 대형 트레일러
  5: 오토바이(자전거)
  6: 보행자


## 4. 이미지 ↔ 라벨 매칭 (폴더명 = 카메라ID)

In [7]:
pairs = []
no_camera, no_label_for_img = [], []

for img_path in tqdm(image_files, desc="매칭중"):
    camera_id = img_path.parent.name
    cam_dict = label_index.get(camera_id)
    if cam_dict is None:
        no_camera.append(img_path)
        continue
    boxes = cam_dict.get(img_path.name)
    if boxes is None:
        no_label_for_img.append(img_path)
        continue
    pairs.append((img_path, boxes))

print(f"매칭 성공: {len(pairs):,}개")
print(f"카메라ID 자체를 못 찾은 이미지: {len(no_camera):,}개")
print(f"카메라는 있는데 파일명 라벨이 없는 이미지: {len(no_label_for_img):,}개")
if no_camera:
    print("라벨을 못 찾은 카메라ID 예시:", sorted({p.parent.name for p in no_camera})[:10])

# 클래스별 박스 수 (불균형 확인 — 발표 자료용으로도 유용)
cls_counter = Counter()
for _, boxes in pairs:
    for cat_id, _ in boxes:
        cls_counter[global_categories[cat_id]] += 1
print("\n클래스별 박스 수:")
for name, cnt in cls_counter.most_common():
    print(f"  {name}: {cnt:,}")


매칭중: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 145694/145694 [00:00<00:00, 447255.96it/s]


매칭 성공: 145,419개
카메라ID 자체를 못 찾은 이미지: 0개
카메라는 있는데 파일명 라벨이 없는 이미지: 275개

클래스별 박스 수:
  승용차: 1,140,144
  보행자: 247,260
  트럭: 162,668
  대형버스: 70,418
  오토바이(자전거): 34,801
  소형버스: 5,355
  대형 트레일러: 1,736


## 5. 카메라 단위 train/val 분리 (수정 B)

같은 카메라의 연속 프레임이 train/val 양쪽에 들어가는 데이터 누수를 차단합니다.
val 성능 = "모델이 한 번도 본 적 없는 CCTV 지점"에서의 성능.

In [8]:
VAL_RATIO = 0.2
SEED = 42

camera_ids = sorted({p.parent.name for p, _ in pairs})
random.Random(SEED).shuffle(camera_ids)
n_val = max(1, int(len(camera_ids) * VAL_RATIO))
val_cams = set(camera_ids[:n_val])

train_pairs = [(p, b) for p, b in pairs if p.parent.name not in val_cams]
val_pairs   = [(p, b) for p, b in pairs if p.parent.name in val_cams]

print(f"카메라: train {len(camera_ids)-n_val}대 / val {n_val}대")
print(f"이미지: train {len(train_pairs):,}장 / val {len(val_pairs):,}장")
print("val 카메라 목록:", sorted(val_cams))


카메라: train 56대 / val 14대
이미지: train 116,246장 / val 29,173장
val 카메라 목록: ['BC1000301', 'BC1000501', 'BC1000601', 'BC2000102', 'BC2000103', 'BC2000105', 'BC2000202', 'BC2000204', 'SC0004601', 'SC0004602', 'SC0004604', 'SC0028804', 'SC0041701', 'SC5863301']


## 6. YOLO 데이터셋 변환 (수정 A·D·E 반영)

- 이미지 디코딩 없이 `size_index`로 크기 조회 (미지 해상도 카메라만 1장 실측)
- bbox는 폐기 대신 **클리핑** 후 사용
- 출력 파일명 = `{카메라ID}_{원본파일명}` (충돌 방지)
- 하드링크 우선, 실패 시 복사

In [9]:
for split in ["train", "val"]:
    (OUTPUT_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

_probe_cache = {}   # camera_id -> (w, h)  : RES_MAP에 없던 카메라용 1회 실측

def get_size(cam, img_path):
    wh = size_index.get(cam, {}).get(img_path.name)
    if wh:
        return wh
    if cam in _probe_cache:
        return _probe_cache[cam]
    img = imread_kr(img_path)              # 카메라당 최초 1회만 실제 디코딩
    if img is None:
        return None
    h, w = img.shape[:2]
    _probe_cache[cam] = (w, h)
    return (w, h)

def convert_and_save(pairs_subset, split, use_link=True):
    skipped_size, skipped_empty, link_failed = 0, 0, 0
    for img_path, boxes in tqdm(pairs_subset, desc=f"{split} 변환중"):
        cam = img_path.parent.name
        wh = get_size(cam, img_path)
        if wh is None:
            skipped_size += 1
            continue
        w, h = wh
        yolo_lines = []
        for cat_id, bbox in boxes:
            if cat_id not in CAT_ID_TO_YOLO_ID:
                continue
            x1 = max(0.0, min(float(bbox[0]), w)); y1 = max(0.0, min(float(bbox[1]), h))
            x2 = max(0.0, min(float(bbox[2]), w)); y2 = max(0.0, min(float(bbox[3]), h))
            if x2 - x1 < 2 or y2 - y1 < 2:      # 클리핑 후 사실상 소멸한 박스만 제외
                continue
            xc, yc = (x1 + x2) / 2 / w, (y1 + y2) / 2 / h
            bw, bh = (x2 - x1) / w, (y2 - y1) / h
            yolo_lines.append(f"{CAT_ID_TO_YOLO_ID[cat_id]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
        if not yolo_lines:
            skipped_empty += 1
            continue
        stem = f"{cam}_{img_path.stem}"
        dst_img = OUTPUT_ROOT / "images" / split / f"{stem}{img_path.suffix.lower()}"
        if not dst_img.exists():
            done = False
            if use_link:
                try:
                    os.link(img_path.resolve(), dst_img)
                    done = True
                except OSError:
                    link_failed += 1
            if not done:
                try:
                    shutil.copy2(img_path, dst_img)
                except Exception as e:
                    print(f"[경고] 복사 실패, 건너뜀: {img_path} ({e})")
                    continue
        with open(OUTPUT_ROOT / "labels" / split / f"{stem}.txt", "w") as f:
            f.write("\n".join(yolo_lines) + "\n")
    print(f"[{split}] 크기 확인 실패 스킵: {skipped_size}개 / 유효 박스 없음 스킵: {skipped_empty}개"
          f" / 링크 실패→복사: {link_failed}개")

convert_and_save(train_pairs, "train")
convert_and_save(val_pairs, "val")


train 변환중: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 116246/116246 [02:31<00:00, 766.58it/s]


[train] 크기 확인 실패 스킵: 0개 / 유효 박스 없음 스킵: 413개 / 링크 실패→복사: 0개


val 변환중: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 29173/29173 [00:28<00:00, 1032.90it/s]

[val] 크기 확인 실패 스킵: 0개 / 유효 박스 없음 스킵: 22개 / 링크 실패→복사: 0개


## 7. data.yaml 생성

In [10]:
yaml_lines = [
    f"path: {OUTPUT_ROOT.resolve()}",
    "train: images/train",
    "val: images/val",
    "",
    "names:",
]
for i, name in enumerate(CLASS_NAMES):
    yaml_lines.append(f"  {i}: {name}")

data_yaml_path = OUTPUT_ROOT / "data.yaml"
with open(data_yaml_path, "w", encoding="utf-8") as f:
    f.write("\n".join(yaml_lines) + "\n")
print(open(data_yaml_path, encoding="utf-8").read())


path: C:\Project\PythonProject\PyTorch001\final_project\yolo_dataset_v3
train: images/train
val: images/val

names:
  0: 승용차
  1: 소형버스
  2: 대형버스
  3: 트럭
  4: 대형 트레일러
  5: 오토바이(자전거)
  6: 보행자



## 8. (선택) 스모크 테스트 — 파이프라인 검증용 2 epoch

본학습 전에 이 셀로 라벨/경로 오류가 없는지 5~10분 내로 확인하세요.
결과는 `vehicle_detector_v3_smoke`에 저장되어 본학습과 겹치지 않습니다.

In [11]:
# model_smoke = YOLO("yolov8s.pt")
# model_smoke.train(
#     data=str(data_yaml_path), epochs=2, imgsz=640, batch=16, device=0, workers=8,
#     project=RUN_PROJECT, name=RUN_NAME + "_smoke", exist_ok=True,
# )


## 9. 본학습 (수정 C: epochs=50, patience=15)

In [12]:
# ==== 9번 단독 실행용 부트스트랩 (커널 리셋 후 여기부터 실행 가능) ====
from pathlib import Path
import shutil
from ultralytics import YOLO

OUTPUT_ROOT      = Path("./yolo_dataset_v3")
data_yaml_path   = OUTPUT_ROOT / "data.yaml"
RUN_PROJECT      = "flood_stage1"
RUN_NAME         = "vehicle_detector_v3"
FINAL_MODEL_PATH = Path("./models/stage1_vehicle_detector_v3.pt")

# 변환이 끝난 상태인지 확인 (아니면 학습이 이상하게 돌기 전에 여기서 멈춤)
assert data_yaml_path.exists(), "data.yaml이 없습니다 → 2~7번을 먼저 실행하세요"
n_train = len(list((OUTPUT_ROOT / "images/train").glob("*")))
n_val   = len(list((OUTPUT_ROOT / "images/val").glob("*")))
assert n_train > 0 and n_val > 0, f"이미지가 비어 있습니다 (train {n_train} / val {n_val})"
print(f"데이터셋 확인 완료 — train {n_train:,}장 / val {n_val:,}장")

데이터셋 확인 완료 — train 398,108장 / val 29,151장


In [13]:
model = YOLO("yolo26n.pt")   # COCO 사전학습 가중치

results = model.train(
    data=str(data_yaml_path),
    epochs=50,
    patience=15,        # 15 epoch 동안 개선 없으면 조기 종료
    imgsz=640,
    batch=16,           # GPU 메모리 부족(OOM) 시 8로 낮추세요
    device=0,
    workers=4,
    cache="disk",
    project=RUN_PROJECT,
    name="vehicle_detector_v3_y26",      # flood_stage1/vehicle_detector_v3 에 저장 (기존 run과 분리)
    exist_ok=True,
)

save_dir = Path(results.save_dir)
best_model_path = save_dir / "weights" / "best.pt"
print("best.pt 경로:", best_model_path, "| 존재:", best_model_path.exists())


New https://pypi.org/project/ultralytics/8.4.90 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.86  Python-3.11.9 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset_v3\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosai

## 10. 검증 — best.pt 기준, 처음 보는 카메라(val)에서의 성능

In [14]:
best_model = YOLO(best_model_path)      # last가 아닌 best 가중치로 평가
metrics = best_model.val(data=str(data_yaml_path))

print("mAP50-95:", f"{metrics.box.map:.4f}")
print("mAP50:   ", f"{metrics.box.map50:.4f}")
print("클래스별 mAP50-95:")
for i, name in best_model.names.items():
    print(f"  {name}: {metrics.box.maps[i]:.4f}")


Ultralytics 8.4.86  Python-3.11.9 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
YOLO26n summary (fused): 122 layers, 2,376,201 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.20.1 ms, read: 785.0692.8 MB/s, size: 483.8 KB)
val: Scanning C:\Project\PythonProject\PyTorch001\final_project\yolo_dataset_v3\labels\val.cache... 29151 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 29151/29151  0.0s
val: C:\Project\PythonProject\PyTorch001\final_project\yolo_dataset_v3\images\val\SC0028804_20200918_130000_S_7050.jpg: 2 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1822/1822 9.1it/s 3:200.1sss
                   all      29151     343402        0.6      0.386      0.424      0.295
                         28689     226979      0.774      0.662       0.73      0.515
                          792        829      0.545      0.286      0.286      0.23

In [15]:
from ultralytics import YOLO

# args.yaml이 아니라 실제 데이터셋 정의 파일을 써야 함
data_yaml_path = r"C:\Project\PythonProject\PyTorch001\final_project\yolo_dataset_v3\data.yaml"

best_model = YOLO(r"C:\Project\PythonProject\PyTorch001\final_project\runs\detect\flood_stage1\vehicle_detector_v3_y26\weights\best.pt")
metrics = best_model.val(data=str(data_yaml_path))

print("mAP50-95:", f"{metrics.box.map:.4f}")
print("mAP50:   ", f"{metrics.box.map50:.4f}")  # 전체 mAP50 (스칼라, 인덱싱 불가)

print("클래스별 mAP50-95:")
for i, name in best_model.names.items():
    print(f"  {name}: {metrics.box.maps[i]:.4f}")

# 클래스별 mAP50을 별도로 보고 싶다면:
print("클래스별 mAP50 (검출된 클래스만):")
for idx, cls_id in enumerate(metrics.box.ap_class_index):
    print(f"  {best_model.names[cls_id]}: {metrics.box.ap50[idx]:.4f}")

Ultralytics 8.4.86  Python-3.11.9 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
YOLO26n summary (fused): 122 layers, 2,376,201 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 3.63.7 ms, read: 115.862.7 MB/s, size: 612.2 KB)
val: Scanning C:\Project\PythonProject\PyTorch001\final_project\yolo_dataset_v3\labels\val.cache... 29151 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 29151/29151  0.0s
val: C:\Project\PythonProject\PyTorch001\final_project\yolo_dataset_v3\images\val\SC0028804_20200918_130000_S_7050.jpg: 2 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1822/1822 10.2it/s 2:58.2sss
                   all      29151     343402        0.6      0.386      0.424      0.295
                         28689     226979      0.774      0.662       0.73      0.515
                          792        829      0.545      0.286      0.286      0.235

In [16]:
FINAL_MODEL_PATH.parent.mkdir(exist_ok=True)
shutil.copy2(best_model_path, FINAL_MODEL_PATH)
print("최종 모델 저장:", FINAL_MODEL_PATH.resolve())


최종 모델 저장: C:\Project\PythonProject\PyTorch001\final_project\models\stage1_vehicle_detector_v3.pt


## 11. 추론 테스트 (한글 경로 대응)

커널을 재시작해도 이 섹션부터 독립적으로 실행할 수 있습니다.

In [17]:
from ultralytics import YOLO
from pathlib import Path
import cv2, random
import numpy as np
import matplotlib.pyplot as plt

MODEL_PATH = Path("./models/stage1_vehicle_detector_v3.pt")
if not MODEL_PATH.exists():
    cands = sorted(Path("./flood_stage1").rglob("vehicle_detector_v3*/weights/best.pt"))
    if not cands:
        raise FileNotFoundError("v3 모델을 찾을 수 없습니다. 9~10단계를 먼저 실행하세요.")
    MODEL_PATH = cands[-1]

trained_model = YOLO(MODEL_PATH)
print("모델 로드:", MODEL_PATH)
print("클래스:", trained_model.names)

def imread_kr(path):
    try:
        data = np.fromfile(str(path), dtype=np.uint8)
        return cv2.imdecode(data, cv2.IMREAD_COLOR) if data.size else None
    except Exception:
        return None


모델 로드: models\stage1_vehicle_detector_v3.pt
클래스: {0: '승용차', 1: '소형버스', 2: '대형버스', 3: '트럭', 4: '대형 트레일러', 5: '오토바이(자전거)', 6: '보행자'}


In [18]:
# 한 장 테스트 — 한글 경로도 OK
def detect_and_show(image_path, conf=0.25, figsize=(12, 8)):
    img = imread_kr(image_path)
    if img is None:
        print(f"[오류] 이미지를 읽을 수 없습니다: {image_path}")
        return
    result = trained_model.predict(img, conf=conf, verbose=False)[0]
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
    plt.axis("off"); plt.title(Path(image_path).name); plt.show()
    print(f"탐지 객체 수: {len(result.boxes)}")
    for box in result.boxes:
        print(f"  - {trained_model.names[int(box.cls[0])]}: {float(box.conf[0]):.2f}")

# 경로만 바꿔서 실행
# detect_and_show(r"C:\Users\Win11Pro\Downloads\test_photo.jpg")


In [19]:
# val 카메라 이미지 무작위 3장 — "처음 보는 지점" 성능 눈으로 확인
val_imgs = list((Path("./yolo_dataset_v3/images/val")).glob("*.jpg"))
if val_imgs:
    for p in random.sample(val_imgs, min(3, len(val_imgs))):
        detect_and_show(p)
else:
    print("val 이미지가 없습니다. 6단계 변환을 먼저 실행하세요.")


<Figure size 1200x800 with 1 Axes>

탐지 객체 수: 7
  - 승용차: 0.96
  - 승용차: 0.96
  - 트럭: 0.71
  - 승용차: 0.58
  - 트럭: 0.40
  - 승용차: 0.38
  - 승용차: 0.34


<Figure size 1200x800 with 1 Axes>

탐지 객체 수: 4
  - 승용차: 0.93
  - 승용차: 0.91
  - 보행자: 0.39
  - 보행자: 0.31


<Figure size 1200x800 with 1 Axes>

탐지 객체 수: 0


In [20]:
# 영상 파일 테스트 (프레임 단위 스트림 처리)
def detect_video(video_path, conf=0.25, preview_every_n_sec=5):
    video_path = Path(video_path)
    if not video_path.exists():
        print(f"[오류] 영상이 없습니다: {video_path}")
        return
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    cap.release()
    interval = max(int(fps * preview_every_n_sec), 1)

    gen = trained_model.predict(source=str(video_path), conf=conf, save=True, stream=True, verbose=False)
    saved_dir = None
    for i, r in enumerate(gen):
        if saved_dir is None:
            saved_dir = r.save_dir
        if i % interval == 0:
            plt.figure(figsize=(10, 6))
            plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
            plt.axis("off"); plt.title(f"frame {i}"); plt.show()
    print("결과 영상 저장 위치:", saved_dir)

# detect_video(r"C:\Users\Win11Pro\Downloads\test_video.mp4")
